In [ ]:
from google.colab import drive
from google.colab.patches import cv2_imshow
import cv2
import numpy as np
import os

# Mount Google Drive
drive.mount('/content/drive')

# Function to process the image, crop the drawing, handle small drawings, and center it on a larger white canvas
def crop_and_center(input_path, output_path):
    # Read the image
    image = cv2.imread(input_path) # Reads an image file into a numpy array for processing.

    # Crop out the gray header (top 20% of the image)
    height, width, _ = image.shape
    cropped_image = image[int(height * 0.2):, :]  # Removes the grey part

    # Convert the image to the HSV color space to detect blue, with this format it easier to detect blue color.
    hsv = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2HSV)

    # Define the range for blue color in HSV
    lower_blue = np.array([100, 50, 50])
    upper_blue = np.array([140, 255, 255])
    mask = cv2.inRange(hsv, lower_blue, upper_blue) # create a binary image that stores the blue parts as '1'S.

    # will find the drawing of the child and will store it as array of np values.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Get the bounding box around the blue drawing
        x_min, y_min, x_max, y_max = float('inf'), float('inf'), 0, 0
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            x_min = min(x_min, x)
            y_min = min(y_min, y)
            x_max = max(x_max, x + w)
            y_max = max(y_max, y + h)

        # Crop the drawing
        cropped_drawing = cropped_image[y_min:y_max, x_min:x_max]

        # Create a larger white canvas
        canvas_size = 1000  # Adjust canvas size for more white space
        canvas = np.ones((canvas_size, canvas_size, 3), dtype=np.uint8) * 255 # will set all the values in the np array to (255,255,255) which represents white color.

        # will return the sizes of the height and the width of the cropped image.
        drawing_height, drawing_width = cropped_drawing.shape[:2]

        # Avoid excessive zoom-in by setting a minimum scale factor
        if drawing_width > 0 and drawing_height > 0:
            scale = min(canvas_size / (drawing_width * 2.5), canvas_size / (drawing_height * 2.5))
            scale = min(scale, 1.0)  # Limit the maximum zoom-in to 1.0 (no zoom)
            resized_drawing = cv2.resize(cropped_drawing, (int(drawing_width * scale), int(drawing_height * scale)))

            # Get coordinates to center the drawing on the canvas
            y_offset = (canvas_size - resized_drawing.shape[0]) // 2
            x_offset = (canvas_size - resized_drawing.shape[1]) // 2

            # Paste the resized drawing onto the canvas
            canvas[y_offset:y_offset + resized_drawing.shape[0], x_offset:x_offset + resized_drawing.shape[1]] = resized_drawing

            # Save the final centered drawing
            cv2.imwrite(output_path, canvas) #Saves the processed image to a file.
            print(f"Processed image saved at {output_path}")
        else:
            print("Drawing dimensions are invalid, skipping processing.")
    else:
        print(f"No blue drawing detected in {input_path}.")

# Function to process all images in the described folder structure
def process_folders(main_folder_path):
    # Loop through all subfolders (children) in the main folder
    for child_folder in os.listdir(main_folder_path):
        child_folder_path = os.path.join(main_folder_path, child_folder)

        # Ensure it's a directory
        if os.path.isdir(child_folder_path):
            # Path to the SimpleTest folder
            simple_test_folder = os.path.join(child_folder_path, "SimpleTest")

            # Ensure the SimpleTest folder exists
            if os.path.exists(simple_test_folder):
                # Path to the "Extracted images" folder
                extracted_images_folder = os.path.join(child_folder_path, "Extracted images")

                # Create the "Extracted images" folder if it doesn't exist
                os.makedirs(extracted_images_folder, exist_ok=True)

                # Process all PNG files in the SimpleTest folder
                for file_name in os.listdir(simple_test_folder):
                    if file_name.endswith(".png"):  # Check for PNG files
                        input_image_path = os.path.join(simple_test_folder, file_name)
                        output_image_path = os.path.join(extracted_images_folder, file_name)

                        # Process the image
                        crop_and_center(input_image_path, output_image_path)
                        print(f"Processed {file_name} and saved to {output_image_path}.")
            else:
                print(f"SimpleTest folder not found in {child_folder_path}.")
        else:
            print(f"{child_folder_path} is not a folder.")

# Define the main folder path in Google Drive
main_folder_path = '/content/drive/MyDrive/sample shapes'

# Run the processing function
process_folders(main_folder_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/children.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/Notebooks.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/shapes.
/content/drive/MyDrive/sample shapes/Clustering Child Drawings: Progress and Challenges.gdoc is not a folder.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/clustering all shapes.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/image processing.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/each shape.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/add new child.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/erode.
SimpleTest folder not found in /content/drive/MyDrive/sample shapes/siamese network.
SimpleTest folder not found 

Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/4.png
Processed 4.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/4.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/2.png
Processed 2.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/2.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/6.png
Processed 6.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/6.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/15.png
Processed 15.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/15.png.


Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/16.png
Processed 16.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/16.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/8.png
Processed 8.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/8.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/9.png
Processed 9.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/9.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/10.png
Processed 10.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/10.png.


Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/12.png
Processed 12.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/12.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/11.png
Processed 11.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/11.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/14.png
Processed 14.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/14.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/13.png
Processed 13.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/13.png.


Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/7.png
Processed 7.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/7.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/19.png
Processed 19.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/19.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/17.png
Processed 17.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/17.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/20.png
Processed 20.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/20.png.
No blue drawing detected in /content/drive/MyDrive/sample shapes/new children unprocessed/S

Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/21.png
Processed 21.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/21.png.
Processed image saved at /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/18.png
Processed 18.png and saved to /content/drive/MyDrive/sample shapes/new children unprocessed/Extracted images/18.png.
/content/drive/MyDrive/sample shapes/Folder Structure and Description.gdoc is not a folder.
